---
# **Práctica 4**: *Evaluación de modelos del lenguaje neuronales*
---
### **Alumno**:  Roberto Samuel Sanchez Rosas

In [1]:
!pip install torch nltk numpy subword-nmt

import os
import time
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import nltk
from nltk import ngrams

nltk.download('genesis')
nltk.download('gutenberg')
nltk.download('inaugural')
nltk.download('state_union')
nltk.download('webtext')
nltk.download('abc')
nltk.download('punkt')

# Variables globales
UNK_LABEL = "<UNK>"
BOS_LABEL = "<BOS>"
EOS_LABEL = "<EOS>"
EMBEDDING_DIM = 200
CONTEXT_SIZE = 2
BATCH_SIZE = 256
H = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Entrenando en: {device}")

# Creamos las carpetas locales
CORPORA_PATH = "./data"
MODELS_PATH = "./models"
os.makedirs(CORPORA_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

[nltk_data] Downloading package genesis to /root/nltk_data...
[nltk_data]   Unzipping corpora/genesis.zip.
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Unzipping corpora/inaugural.zip.
[nltk_data] Downloading package state_union to /root/nltk_data...
[nltk_data]   Unzipping corpora/state_union.zip.
[nltk_data] Downloading package webtext to /root/nltk_data...
[nltk_data]   Unzipping corpora/webtext.zip.
[nltk_data] Downloading package abc to /root/nltk_data...
[nltk_data]   Unzipping corpora/abc.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Entrenando en: cuda


In [2]:
def lm_preprocess_corpus(corpus: list) -> list:
    """Agrega tokens de inicio y fin, normaliza a minusculas"""
    preprocessed_corpus = []
    for sent in corpus:
        result = [word.lower() for word in sent]
        result.append(EOS_LABEL)
        result.insert(0, BOS_LABEL)
        preprocessed_corpus.append(result)
    return preprocessed_corpus

def get_words_freqs(corpus: list) -> dict:
    """Calcula la frecuencia de las palabras en un corpus"""
    words_freqs = {}
    for sentence in corpus:
        for word in sentence:
            words_freqs[word] = words_freqs.get(word, 0) + 1
    return words_freqs

def get_words_indexes(words_freqs: dict):
    """Calcula los indices de las palabras dadas sus frecuencias"""
    result = {}
    for idx, word in enumerate(words_freqs.keys()):
        # Para evitar un vocabulario inmenso, mandamos a UNK frecuencias bajas
        if words_freqs[word] <= 2:
            result[UNK_LABEL] = len(words_freqs)
        else:
            result[word] = idx

    # Remapeo limpio para evitar saltos en los índices
    word2idx = {word: idx for idx, word in enumerate(result.keys())}
    # Aseguramos que BOS y EOS existan
    for special in [BOS_LABEL, EOS_LABEL, UNK_LABEL]:
        if special not in word2idx:
            word2idx[special] = len(word2idx)

    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

def get_word_id(words_indexes: dict, word: str) -> int:
    """Obtiene el id de una palabra dada"""
    unk_word_id = words_indexes.get(UNK_LABEL, 0)
    return words_indexes.get(word, unk_word_id)

def get_train_test_data(corpus: list, words_indexes: dict, n: int):
    """Obtiene el conjunto de train y test"""
    x_data = []
    y_data = []
    for sent in corpus:
        if len(sent) < n: continue
        n_grams = ngrams(sent, n)
        for gram in n_grams:
            contexto = [get_word_id(words_indexes, w) for w in gram[:-1]]
            objetivo = [get_word_id(words_indexes, gram[-1])]
            x_data.append(contexto)
            y_data.append(objetivo)
    return x_data, y_data



def calcular_oov_rate(corpus, words_indexes):
    total = 0
    oov = 0
    for sent in corpus:
        for word in sent:
            total += 1
            if word not in words_indexes:
                oov += 1
    return oov / total if total > 0 else 0

def evaluate_perplexity(model, data_loader, device):
    model.eval()
    total_loss = 0
    total_words = 0
    loss_function = nn.NLLLoss(reduction='sum')

    with torch.no_grad():
        for data_tensor in data_loader:
            context = data_tensor[:, 0:2].to(device)
            target = data_tensor[:, 2].to(device)
            log_probs = model(context)
            loss = loss_function(log_probs, target)
            total_loss += loss.item()
            total_words += target.size(0)

    return np.exp(total_loss / total_words)

In [3]:
# Trigram Neural Network Model
class TrigramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, h):
        super(TrigramModel, self).__init__()
        self.context_size = context_size
        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, h)
        self.linear2 = nn.Linear(h, vocab_size)
        self.dropout = nn.Dropout(0.2) # Agregado para mejorar el modelo

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((-1, self.context_size * self.embedding_dim))
        out = torch.tanh(self.linear1(embeds))
        out = self.dropout(out)
        out = self.linear2(out)
        log_probs = F.log_softmax(out, dim=1)
        return log_probs

def train_lm_model(model, train_loader, epochs=4):
    loss_function = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.002)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

    for epoch in range(epochs):
        model.train()
        st = time.time()
        for it, data_tensor in enumerate(train_loader):
            context = data_tensor[:, 0:2].to(device)
            target = data_tensor[:, 2].to(device)

            model.zero_grad()
            log_probs = model(context)
            loss = loss_function(log_probs, target)
            loss.backward()
            optimizer.step()

        scheduler.step()
        print(f"Epoch {epoch} terminada en {time.time()-st:.2f}s | Loss final aprox: {loss.item():.4f}")
    return model

In [4]:
from nltk.corpus import abc, genesis, gutenberg, inaugural, state_union, webtext
nltk.download('punkt_tab')

print("--- 1. PREPARANDO CORPORA NLTK ---")
# Juntamos los corpus para entrenamiento
corpora_train = []
for corpus in [gutenberg, inaugural, state_union, webtext, abc]:
    corpora_train.extend(lm_preprocess_corpus(corpus.sents()))

# Genesis como test
corpora_test = lm_preprocess_corpus(genesis.sents())

print("--- 2. ENTRENANDO MODELO BASE (WORD-BASED) ---")
words_freqs_base = get_words_freqs(corpora_train)
words_indexes_base, index_to_word_base = get_words_indexes(words_freqs_base)

x_train_base, y_train_base = get_train_test_data(corpora_train, words_indexes_base, n=3)
x_test_base, y_test_base = get_train_test_data(corpora_test, words_indexes_base, n=3)

train_loader_base = DataLoader(np.concatenate((x_train_base, y_train_base), axis=1), batch_size=BATCH_SIZE, shuffle=True)
test_loader_base = DataLoader(np.concatenate((x_test_base, y_test_base), axis=1), batch_size=BATCH_SIZE)

V_base = len(words_indexes_base)
model_base = TrigramModel(V_base, EMBEDDING_DIM, CONTEXT_SIZE, H).to(device)
model_base = train_lm_model(model_base, train_loader_base, epochs=3)

ppl_base = evaluate_perplexity(model_base, test_loader_base, device)
oov_base = calcular_oov_rate(corpora_test, words_indexes_base)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


--- 1. PREPARANDO CORPORA NLTK ---
--- 2. ENTRENANDO MODELO BASE (WORD-BASED) ---
Epoch 0 terminada en 107.21s | Loss final aprox: 5.9795
Epoch 1 terminada en 108.89s | Loss final aprox: 6.0530
Epoch 2 terminada en 112.57s | Loss final aprox: 2.9965


In [5]:
print("\n--- 3. ENTRENANDO MODELO SUB-WORD (BPE) ---")
# Guardamos en texto plano para subword-nmt
with open(f"{CORPORA_PATH}/train_nltk.txt", "w") as f:
    for sent in corpora_train: f.write(" ".join(sent) + "\n")
with open(f"{CORPORA_PATH}/test_nltk.txt", "w") as f:
    for sent in corpora_test: f.write(" ".join(sent) + "\n")

# Entrenar modelo BPE y aplicarlo
!subword-nmt learn-bpe -s 10000 < {CORPORA_PATH}/train_nltk.txt > {MODELS_PATH}/nltk_bpe.model
!subword-nmt apply-bpe -c {MODELS_PATH}/nltk_bpe.model < {CORPORA_PATH}/train_nltk.txt > {CORPORA_PATH}/train_bpe.txt
!subword-nmt apply-bpe -c {MODELS_PATH}/nltk_bpe.model < {CORPORA_PATH}/test_nltk.txt > {CORPORA_PATH}/test_bpe.txt

# Cargar BPE a memoria
with open(f"{CORPORA_PATH}/train_bpe.txt", "r") as f:
    corpora_train_bpe = [line.strip().split() for line in f.readlines()]
with open(f"{CORPORA_PATH}/test_bpe.txt", "r") as f:
    corpora_test_bpe = [line.strip().split() for line in f.readlines()]

words_freqs_bpe = get_words_freqs(corpora_train_bpe)
words_indexes_bpe, index_to_word_bpe = get_words_indexes(words_freqs_bpe)

x_train_bpe, y_train_bpe = get_train_test_data(corpora_train_bpe, words_indexes_bpe, n=3)
x_test_bpe, y_test_bpe = get_train_test_data(corpora_test_bpe, words_indexes_bpe, n=3)

train_loader_bpe = DataLoader(np.concatenate((x_train_bpe, y_train_bpe), axis=1), batch_size=BATCH_SIZE, shuffle=True)
test_loader_bpe = DataLoader(np.concatenate((x_test_bpe, y_test_bpe), axis=1), batch_size=BATCH_SIZE)

V_bpe = len(words_indexes_bpe)
model_bpe = TrigramModel(V_bpe, EMBEDDING_DIM, CONTEXT_SIZE, H).to(device)
model_bpe = train_lm_model(model_bpe, train_loader_bpe, epochs=3)

ppl_bpe = evaluate_perplexity(model_bpe, test_loader_bpe, device)
oov_bpe = calcular_oov_rate(corpora_test_bpe, words_indexes_bpe)


--- 3. ENTRENANDO MODELO SUB-WORD (BPE) ---
100% 10000/10000 [00:12<00:00, 823.60it/s]
Epoch 0 terminada en 45.80s | Loss final aprox: 5.0957
Epoch 1 terminada en 45.90s | Loss final aprox: 4.8126
Epoch 2 terminada en 46.58s | Loss final aprox: 4.9299


In [8]:
import random

def get_likely_words(model, context_str, words_indexes, index_to_word):
    model.eval()
    model_probs = {}
    words = context_str.split()
    idx1 = get_word_id(words_indexes, words[0])
    idx2 = get_word_id(words_indexes, words[1])

    with torch.no_grad():
        probs = model(torch.tensor([[idx1, idx2]]).to(device)).cpu().detach().tolist()

    for idx, p in enumerate(probs[0]):
        model_probs[idx] = p

    return sorted(((prob, index_to_word[idx]) for idx, prob in model_probs.items()), reverse=True)

def get_next_top_p_word(words, p=0.7):
    if not words: return EOS_LABEL
    valid_words, valid_probs = [], []
    cumulative_prob = 0.0

    for log_prob, word in words:
        prob = np.exp(log_prob)
        valid_words.append(word)
        valid_probs.append(prob)
        cumulative_prob += prob
        if cumulative_prob >= p: break

    return random.choices(valid_words, weights=valid_probs, k=1)[0]

def generar_texto_bpe(model, contexto_inicial, words_indexes, index_to_word, max_tokens=20):
    texto_generado = contexto_inicial.split()
    palabras_ensambladas = []
    buffer_subword = ""

    for i in range(max_tokens):
        context_str = " ".join(texto_generado[-2:])
        siguiente_token = get_next_top_p_word(get_likely_words(model, context_str, words_indexes, index_to_word), p=0.8)

        if siguiente_token == EOS_LABEL: break
        texto_generado.append(siguiente_token)

    # Lógica para pegar los sub-words ('@@')
    for token in texto_generado:
        if token in [BOS_LABEL, EOS_LABEL]: continue

        if token.endswith("@@"):
            buffer_subword += token[:-2]
        else:
            palabras_ensambladas.append(buffer_subword + token)
            buffer_subword = ""

    return " ".join(palabras_ensambladas)

# ================= IMPRESIÓN DE RESULTADOS =================
print("\n" + "="*50)
print("ANÁLISIS COMPARATIVO")
print("="*50)
print(f"Métrica                 | Modelo Base | Modelo Subword")
print("-" * 50)
print(f"Perplejidad (genesis)   | {ppl_base:<11.2f} | {ppl_bpe:<11.2f}")
print(f"Tamaño vocabulario      | {V_base:<11} | {V_bpe:<11}")
print(f"OOV Rate                | {oov_base*100:<10.2f}% | {oov_bpe*100:<10.2f}%")
print("="*50)

print("\n=== GENERACIÓN DE TEXTO CON SUBWORDS ===")
ejemplos = ["<BOS> the", "<BOS> in", "<BOS> and"]
for ej in ejemplos:
    print(f"[{ej}] -> {generar_texto_bpe(model_bpe, ej, words_indexes_bpe, index_to_word_bpe)}")

# ================= GUARDAR MODELOS LOCALMENTE =================
os.makedirs('modelos_finales', exist_ok=True)

torch.save(model_base.state_dict(), 'modelos_finales/modelo_base.pt')
torch.save(model_bpe.state_dict(), 'modelos_finales/modelo_bpe.pt')
!cp models/nltk_bpe.model modelos_finales/nltk_bpe.model

print("\n[✓] Modelos guardados en el almacenamiento temporal de Colab.")


ANÁLISIS COMPARATIVO
Métrica                 | Modelo Base | Modelo Subword
--------------------------------------------------
Perplejidad (genesis)   | 120.97      | 1865.29    
Tamaño vocabulario      | 32631       | 9969       
OOV Rate                | 38.57     % | 3.96      %

=== GENERACIÓN DE TEXTO CON SUBWORDS ===
[<BOS> the] -> the only son of hewed foam in to the right and turned to the american , and felt its responsibility
[<BOS> in] -> in our days , and his arrival , and which in a moment he was in the murray valley .
[<BOS> and] -> and what i am coming up to him .

[✓] Modelos guardados en el almacenamiento temporal de Colab.
